# 무노균병 개선 실험: 학습 크롭 제거

`RandomResizedCrop(256, scale=(0.7, 1.0))`만 `Resize((256, 256))`로 교체합니다.
좌우 반전, 15도 회전, 색상 증강, 정규화, 검증 전처리와 데이터 분할은 그대로 유지합니다. 회전 증강에 의한 모서리 잘림은 남아 있습니다.

- 기존 데이터: Drive의 `cropy_full_512/cropy_full_512.zip`을 그대로 사용합니다. 다시 업로드하지 않습니다.
- 기존 최고 모델: `cropy_runs/full512_resnet50_v1` (읽기 전용 비교 기준).
- 새 결과: `cropy_runs/full512_resnet50_full_resize_v1`.
- ResNet50 / 256px / batch 16 / 20 epoch / seed 42 / 기존 클래스 균형 샘플링. 기존 모델을 미세조정하지 않고 동일한 ImageNet 사전학습 가중치에서 시작합니다.
- 학습 코드를 이 노트북에 포함했습니다. 별도 코드 ZIP 업로드는 필요 없습니다.

GPU(T4) 런타임에서 **모두 실행**합니다. Drive 연결 화면이 나오면 본인 계정으로 연결합니다. 중단되면 같은 노트북을 다시 모두 실행하면 마지막 완료 epoch부터 재개합니다. 설정을 바꾸거나 기준 모델 폴더를 새 결과 경로로 지정하지 마세요.

현재 수치는 검증 성능입니다. 검증셋을 이미 분석했으므로 새로운 독립 테스트 성능으로 해석하지 않습니다. 무노균병은 24장이므로 한 장 차이가 약 4.17%p입니다.

In [ ]:
from pathlib import Path
import base64, csv, hashlib, io, json, platform, shutil, subprocess, sys, zipfile
from collections import Counter
import torch, torchvision
from google.colab import drive

assert torch.cuda.is_available(), '런타임 유형을 GPU(T4)로 변경하고 다시 실행하세요.'
drive.mount('/content/drive')
UPLOAD = Path('/content/drive/MyDrive/cropy_full_512')
WORK = Path('/content/cropy_full_resize_ablation')
BASELINE = Path('/content/drive/MyDrive/cropy_runs/full512_resnet50_v1')
RUN = Path('/content/drive/MyDrive/cropy_runs/full512_resnet50_full_resize_v1')
assert RUN.resolve() != BASELINE.resolve(), '기존 모델 폴더는 새 결과 경로로 사용할 수 없습니다.'
WORK.mkdir(parents=True, exist_ok=True)
EPOCHS, BATCH_SIZE = 20, 16
baseline_config = json.loads((BASELINE / 'config.json').read_text(encoding='utf-8'))
baseline_env = json.loads((BASELINE / 'environment.json').read_text(encoding='utf-8'))
assert str(torch.__version__) == baseline_env['torch'], f"PyTorch 버전 차이: 현재 {torch.__version__}, 기준 {baseline_env['torch']}. 같은 환경을 확인한 뒤 실행하세요."
print('GPU:', torch.cuda.get_device_name(0), '/ 기준:', baseline_env['gpu'])
if torch.cuda.get_device_name(0) != baseline_env['gpu']:
    print('주의: GPU가 달라 수치 차이를 크롭 효과만으로 단정할 수 없습니다.')
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)


## 동일한 데이터와 실험 코드 준비
ZIP과 모든 이미지의 SHA-256을 확인합니다. 학습 이미지는 기존과 동일한 긴 변 최대 512px JPEG이며 모델 입력은 256px입니다.

In [ ]:
EXPECTED_DATA_SHA256 = '52dea0080285d76b6cbc09c2659c052a18f20ef620d05bca462203e64e6872ee'
EXPECTED_MANIFEST_SHA256 = '1d8924f17fa8cc02f69607a14a60e879301e24e2fd74a8a01792b491d8701b34'
EXPECTED_CODE_SHA256 = 'c4a7568a37e5ca2408bbb3a558ac4656331433ad37d8022cc805f4a365a7edf0'
CODE_B64 = 'UEsDBBQAAAAIAHNlRF1moWWjFQsAAAcfAAAIAAAAdHJhaW4ucHnFGWtv29b1u37FHYvBZELRkhtnqTAVy5wE25AFheOsHzyDuKauJM4UyfJSThzNQLK6Q5AaaILVSwrEgQPk8cXFssbrPKD5Qxb1H3buk6QkK0E3bAYSiZfndc/7HBmGUfGSKN5CVbR07Roa7X2T3f8Hyu4/H909HL58O7p/XKncoLhDGhUEf/FW2o1ClCbYDx2GVW3hFFdbfoIcZ5595/9RksIrEkdel6KFGkf9AGUHe6Nv9oZf7aCT10+ynQM02t3N9o9H958Mv9w3sz/vZvceo9Hdo+HTxyAAOvnnwejxHjqLTt4cjPaOss/vAqqNTo5fZ8+OUPbdEYg4+uqQUzjOHj3Idp6gbP8o++uRuIX1I0TGiddFCaEhSRdr6PecAPurVuGw3yPI6xJvI478MKXz64Smbi9qkcCJ0y7ArOMAhx5pVSnuxQFJSgT8XqdK/dsELSyeh8cgQQukupirqb4A36N+GvfTU7i47ZQxqhhgtIrfi6MkRTjpxDihpNJOoh7yoiAgXupHIUUSYCnqhylJxPsYp93AX1fvPoFHTSmN4O6lBycMEaYoDMunUZz6PfaCfxGExZt+6gfUYdpUHC7B96sRbpHERp8Sv9NNSWsZh62od12oqIC+6VMQXGFKm1Ab8bvDJ1gwpO0o6dFKpdIibbTe94OWUI0Z9nuuF2BKCW0g0JuNmCkbiKYJaiJDmLR+wbBQ9WO4kfPbqNUPpFP7bQ6MmhpwsWY0tOk4AyAi5HAUhHmT34c25fkyodfYuSvuSZ1Ll69cvHF1xeKEAIK8g2T9wnSS9QvTSQrPa3tABy501Q8JTkx16Pih2yY47QNtGxW0I3ATAm9CQUJqs0NSN1exCe7qMneV2jxjiwhyOyTqkTTZylXLzemyLGLIkPsAsTjfvYOyR4fZqzvDFz+g4cMHEMaIZZVn3472HkMoD5/to+HLw+xPhyxqX97jkf38TrZ/R4IN3xwNv3yOsr23J6932NnJd7tSeh5ITZSAc7e0qOgMqjv1c5YyallgYd6CsLk5cpCCkznCT5c5r9YSYGhGNqIeDkjTrDk/s4FnzVJGns613Q8CVwj9bq4czCzwUt+saZ4EzChBv8NBn1xOkigx28aNcCOMboYT9hqUD7YNQU+cpu2yGEsRhCEl5uqEuLY+mdDVr6LEvx2FKQ6uBH5sWjNAl6MUszxl1hengy1FQZT8xk8hd5nrCXP/kFDarDkf2pDmQoCkqXiizMs5MXhemE5tJVohIQX1TH99Df7HAdP7as05d2HRRvCxeJ5/1M6v2QiOFxY+Ys8LC+fEx+KapLUm9LiJg/fQ4nta+v91CZkXlE/Y8lYyRfTwBnFVkXNlkTNlpuaZdWqOF94KVUvX9uGbHTT8y9+zpy94pnhxkO3vQLL4dvivezwxPEHDV0+z71+j8abAHB4eDb84htZg+OYhI2Gx5CI7hezpPcgQsodxWJVkfD1WACnYRVZCM0WgL+RCQoPUpuqMI24j0yPPla7IxoA58BosytE8CjmuBxmV4QrSjp8SSJjWNkcVdCQuY7taJLaars3ivlY0wlRdmmX6IrVL7GZAwrH3lg3U4gB7pEfCtLmS9IkFxvyFrO+R20lwy7S4dQnYuo9TImqIjQJZt1tk0/dIuYg0ChWI4ZlSb1GSQP/BggDim1U40fvFJBGobg6xWluDhF0gOgapKEyHY0r0e9CXggoCvA5JkalTiJxnxzGIpjxw0sgUt7LUu8KRxo4T0qKqSpsC1XKg4+rhW2Y9h2OyQImMmQS3Ie9pioFPUwhWQUg/W7l8U27MPORsE9VPgZH6k1BQmsGdocTEVgle6G6CjFL+FEzNgWlcI5n1Wg38/swUAfw1CIdx2f01ixXeyWNetSDj1EpK85nGIIF1iDnRoZRCQYmhLjAvbmjnnGSGEomLd9TjbuwlEKcJVAlbNK6QKLVzl/yZ05AOzdm4QcT1osTPfbiWA+mn/4Vn6gs48E8HsX7Lh4gJz9Xv5XW0QkyJoDiWIZ117G3cxEmJg+ZPUxIXXhT0BU7G0Vl2LEC4Ihx8DzIbCCFZO2MhpR1Y6oBXylr+vuDJmppDPpOhZzkwq5lWkbUqbLl884ilS2EXUPYpLqYrH3MJOVCyeYu1vmr2ci4mnT7Lr5/wN5KjAHNwq+Vi+d408uHTgAqwFZMmdNEsR3/W9+EePEHbqEuCuGn8mpntShSAfNDoRikyuWfOQ7pFtL8ONKhlzOQlJkvFiffxcBncD9LmQm0m5jpOvS4fV6dif7gwEzvQt2uDfgt4dVI9NxNTuENJNwrXOH30nq0FMblPpXktCrW6lzR5sDyS4z6bTmdTZ5PjdHn1zAm5pxtB/NLmavEwnzXXZnKA1Fi9GSUbJJluytmWVGuH6V7A+kBx/ezpF8MDaMfUogWZw78dZzvHCBqt7NVe9uhrvrg4+X4XurZ3+N34HgR4Y76VaBoUOg8Cc2YfBCrVpuKfkCjfBOnOEJ28vpM9f4iyzx+Pvt4dvrqbPfmBdYun7I6klCAay3dSWP7BxIWWrcLfs5B02T6oyTciJnvnqMPykGTnbf7YvMyRdANfKWDxHkKtM5xCVJvgL6ZmPo8MDm9YhVVHUzHOZ4z3JwfQZWJCdikc33p0WHov9/N5jVJbrOYpjb+6XiFt8xORUwEt3/5oWBvx1MK11JT81bOtODbl5+kekv+xxkFGh6BXOJg2MP/HInb77TbM/jxT/1fk2+QVaVIeYez3k+YKhkva72AmnJ23ErxDZyOAeDQNr9/CBl+a8GP26PjUxZvYh5oaENMSLZzhxX0ZVoWWDcixYqo06JQ6ObXsOmVdZws3ZHnUKvY6otVSPYpYci0lULkvw/AfxVtX4auo9y7tQXns+mGnWXNkE6E7FNZjsO/OxRbufSr3Y5ADcI8AZcra8yARygoge4u5yW0RD28VyhWF4sMWhjm5IHH1Icz51A/JxTAkMICHnavLZqHFXHGhuREcREGWdxO1zmU72VLmEefWOAwTGvKr09uA8DbFA5UNA7kFk4UbbcgBj2HyCok9T/atIl0nrO1vG5e4ihtoIHS9baMltTodFCyznW8k4EUpwcEro+D8bWMii0iE8XO2dSqlH1Fo8/iEGpxq72RhYRbAbMhEMQSLJ9Y9420x0GToDqRm05CFubj6/Ukz97XyECbWaNe3oJ3tXb7lMyVdZCvhnk97LNwahZU8Gmgmc4zSnI3mFJM5axttUvVDwkBzU8u2gpSrhlSzscYEG4+dd8rHTTZDwNU5SWlujcukfuQYjHMqyibCg6ndpSlOictaa1OIK36DyI9V21L0Nhu5YLzxTUKe4aZvEwoTt3DQZW7tFm+/pBaFA2wjnhaBEziYYtpw6u3tn1rKs9gAxgMtny6LwTdZCihVy20RL1OGyGLVeMcoqclLQe3SbP2jVZMnm7GR6ybBG2wPj/6IDOcPYH+z5DltY+BvNwZaBhjGG04N9GXMGsFLUzyb33+OPipO75Mmu8w1PhCKP4vq2/ODgta3UZVPg3oLzR+cc+2xRCIk1sbQ4NrKNsrtr77xF+NUziIQCiEz23ubHTxAA6ambYuXOK4xUc0MSzqNDExJEn2sPboch4WsKkHH9i4sa1G8SczBRG8wGUANGW+FULMmewqdJxoTSWIKcMGKgFCss5OwPEU28pw4BUQlfAWml9UlULBioVKVd1HKQRC6DnppcRXKnsA8xZY6lBXuCrs3FFaYzlkVSYmDfsnIsFEYsPoJ9rYmEgIjUgGbum4Ixd51+U8xrstGedeVv8OIub7yb1BLAwQUAAAACABzZURdAFdRbQQMAACAIgAAEwAAAHRyYWluX2Z1bGxfY29sYWIucHm1Wllv3EYSftev6FWwGA4yQ2scZ9fHMoDh2MECXieIHeRBEBotsmemV7zCbsoaC/rv+1UfZHMOOdjF6mFM9lFXV9VX1fT5+fm7vixZs16rXIlyqdtSGWY6oWpVb9hnZbZMtk2+XV43fV2Ibsc6qftKMlEX7F8i7xr2bsW0LGVuVFOn5+fnZ6pqm84w0W1a0WkZ3rdCb0t1HV7/rZs6PHeg1lRn666pWCsMLWN+6he8noV1dV+1OyY0q9swZJou37qd9jHtjSp1WggjAokf8fy+EYXsFux3qTZbI4tfLcePompL2UXbb5WGGmEnUdHS6AWrmkKW+syvJPuENRtpOAZqvW66CivtJLdGi1bzTdf0Lb8GuVLVMmyWt6LshZELmFAW/HPT3UCcs7NCrpkWt5IL01QqT2gZFpFt5i/PGP6MJAp0IJkdTumsuO5xkneJHXDP7Fs2S03VzuZum7URkQ40B0LzKd20k20pcmmJzb1MPG/qtdpwy8wrJptKmm6XuCkvHvzgTVPh/CUr5UbkOwZXaVnX19p6VdMblm9FvSE3M1upOqYhmyyYI9N3YvAnoudGoat7SPOm3SXzaCrFOUFC0ZcmmU0lmy3YzHkYJyG8JTpp+q72u716FbYlXoMWzIILp6+7DZy+Nr/QW+cZt6koCi78VDJbLslfloXqwNHsWpmR8y7A6Y9eQbXsU0cG38qyzWbetZ6oSmyknp2iCDu1vfkKzVObrRPqsFHVZsG8ibKnF6c2XQuDcNfqizy6cfW3UxsRnEvnwCdYntqoqs1pfk+/P8nQnvIyOuV826hc6uxyctw4/TWSHEfiIiZXI+0jXnHIRSBgprSlrqVZPbduZZ+/v5hSDYOnSLoUiv3CJs1sZl0f8dTL02KUZfN5mbf98W3Brd43uShZq1qXZvKtzG80a+pyd5KyrpobuWxlt8xLoY+f3UWgf8F6LTWDNMx57itWN/UXCRxQmn34+RMTjKxNkX6SYyXuvG8uzVbpJa19lOtHg9yRl1JADybWRnbsAwNVRXaAvt7RHT9BGTG1cUtctY9WqCYJlVK3mP3ALiyIidR6PCfnwODKD1qrcFiFW6uwHzJ2YemoNTT2uJPmfSFSpbm4FaoU16VM5nY/rRCpPTOOM3MZxWYdobRkH3caefbtnYI5fvrlNzKWUQDVENUp+2gRFbq8aUCX0aJrCYSRtJawOfXaFvIWfgmVZyTLjMQ7LRogTGIhnCioErZnbngU1AMFwA1xzc22k6LQiY9hFzcpYVbyzI/VbXps2NGpRN2Lkh+ZuRb5jawLTfLWdXot63xbie4GGr0TEPf00kLCDyoUKtqoHMspEzqTeNTmrQCsZOzyyg7DeszWNxajHV4z4PgX1SYeMiikAYuz+WIP1hOYstpYHwkIH/JOJtLpwHw+WrHQ4B/kSf9JEfOuKVGJgCANcyR29uRArGx4mg+kcFgHXjkycswA7FTQENNL92i1zklNUNzI5MXc2cFNYvhSuwF6jgiAm75cXZFj5FeXLw84X13ts4Z9rLIDjf0FBsEIK5BsxvLk0HjKdiQ6OcNUtC0OPim09xxr7wLFFg6LT2xs18fxHtamVmwyTeZ3DSMUsKWsk/2lc1r7Ig76Rw7As0tiQnAjenXs5pZc8t3T71ffLdizi797BF4jmGXXdsh80MMXyaneCiBfErkJZTTkT3jLDNGk1lKbNNe3s3lKocmvd0Yi1c3TrbwrFBKzmZRHZCKVm4TQDB5L/yx82sxCSlywMRNmcVpcsOD8WRQHk/P1SePxyDi2hVJC9uwpLNVlK7l8RkKUos5RDTun6HzZNPQY2WyLGh4K0jmqwtaJqNwA4ny9esWk6EpFs0YBFo9wDNbjzsZZdACAeXf02b4vgP/05LMDXzhtEIeU2Z5rkFtEM95LFtTk+EaAmERvHsdSVxNan6hu8Js4z9DeUPIOGZE3N1FpCPEMpzqeatpoPzmTnbN1QtvABGlrtrOpz0cdn6h3yUQAhRxMIsRJ7xv2Gs6m4Xg1YZjHLRT58PUOB5M3dK5owpwDslzU1FUiPyAvUMdk8+QHadKBZC0qm9Xu1yk92vSxptxxXJiHYaPvV07o7lsHakVnI0Q7UQgyscVx/gdYT1YDKGR9q7qmpqJmGDtiTNt8PVjrRcKk9pRCbRKZO+ZOOfKRhouYpmVDuBxTtunAyDuTAEubAt6dzXqzXj6fzZGD/hJaqCl2HNYlP1tL4SA7HGKDPhP1HQmI/tDsUvYblodKlgQVrJafqTgZd4QSBQoZMqVDcRI4GRwSnbVoeYmq1YW2K3A/2y5dc6pbnRs7+PPsbBnzoall8FPL4SAXP2Y7u+PSH+nsyuZm94JT/NWxmXSiT+yNQqV0RWnxFbuRsrU+3XQKTSyKUMCPgbF16lzpulcE9NDb3R2koS1wqlAGtmXX0C04tSaLV88tKW+OkRQE/EB7uLvO0OmPb9+9/u39pz9F2m1ePd/fbFnZNWDkpU8864ysPVjakfNT83Ffus6HY0aB9h4diOiSMJXC/msp0HVTOn0R7zNN4gpRN9i0qIYBMd1AzY6krwtR/e7pIechMhHwiKEIPJxQvJC52Nkh74KIyaIvDyiWHR+m0jeNhsSvkbWAK/Xm/a/JIMiCfeLoWgaodFQ3spbwjmak+lMYSeZHa968QaVPB4lq+VrV9nWsO3yFhJhQNSBhY7bZi0nzgl3oKRJPBR2M184BJcgevd5KLlfpBdKd23Zprmz6tKXXPuurBdtDKX8LRFnO48ugdTY8xWVZaW/aIMt47ZaMBdvJCiOAvf/3ERg8hrPxn1vG0RgYvq6z6GLtqOwLdMs1r2SFjJXF7RC1U04xguZjajnIfkypbb9elzKzncx/rdJjCn1VdvTXlIshJIDF/XIbxAuPxtQfLdgSLrKwOXXBVifSqgs8MoSjwG1F6TOpnYzGkVOHjUMYndo8LIh3jYF5YtewIN71J/T12/1KurrxI7R475VbvaJBG/+zK/atN5MDT9/3GrsoyNfuzLapeVdvYgHHxhoTfLLBTu2vHxzV7pmsdz65v2GvqydXmGJ9dD8wEYNTbgnAiNmYcPJY6TRPP3eovVzRYauSoq/aUJWgIq01kj4XOlcqxIKqC0opT+c0vVemHOd4UG+dYmt9xCqZadO5p5TzWwQagJzz+cIbyIfMGH2bts8i69AdgFvCqQxMLubHjDuWI/OTyuQkaEeNSoSPb7pG67e16Zp29x6PqIquKYaqpjFbInGRroYeURAqAxeSsVtz/vwtEB/Q5L44aE4XapzKsCVb+arp2KyVOZBy13TUACXr2b3T7+Gly+fZ/QQQHmzf4gZ9x/IwNJL3VqKHNL23Aj+gkluXvd5GnQghj6v6hysJu2nhdURYRZ1ECavQdWeOIiynLx3RxxVXBywmqLMYDb1gEXjHxQX9jXCq9JryqkyIFY5vhnNcuiHLfdIW9CXk5BAjfLYJMowAccjLp5nhDoOc0yqQ2d9RAfSUTl83ELTOBvWfsNXFhbO/E2Y+CXo3hkzsW2EkqR9s/ptGP41AhcPVB6t4KNytzPsJPru/eclu6RZO4DDmKSp3+i0RB4m7ZLqBrHTMDjei/D2nTq1C4fbwNSyfXFec7s0JSsPsiz99WxH/TU6EjpPaBJXrLBy7y2WZ/8h1BKTQObVROzex3YT6gSEP7RM5bzaC52TFcX0GcbJYsJjwRI9F8M5sAptZhJ1WuCyC0eOMR6zLPBpuAhqC6QBsPrtuJuhH9fuAZGNhZleFFce5BpTaT9pTSHska1/G9Ub0vdVXDEObOq7yafL8rU1i9/ZMH57ch1SKtGkj+Z5+X6bP1iFhhngL0Ty7epk+/evDYri5GpeMIWkJnB9mUfr7hr29o2/ImtFHVrTcTQ2063Oj6OrAfXlGe0oqDDcubLya8J9R8CS7rm/tx9ajpogrqH1IHj3EXhuNwh2Ad8iCjwL34IRfLQ58Ao+ylL+dGG4F/icV/t/iH9CPMs7jPEZVLuNNVKB+lWtA+I8wTQHn6Jp+4/97B7t3NM/t2/nVwyuHE+F/eGT3FkjIHVPYmBma9ChInrOH9GdniDhuiybObcxxTt/WOfelqPvQfvYfUEsDBBQAAAAIAHNlRF0BnRJSoQkAAGkbAAAXAAAAdHJhaW5fZ3JvdXBfYmFzZWxpbmUucHmtWW1v3DYS/u5fQRgIJDlaxrt1i8Y9FZdL3aC4XBC06eWDzxBoidrlWRJVkvJb4P/eGVKUuK9xD7cfvEty3jiceWZIHx8f/9KwJf/AzUy29QO5ZprXouWkkoqYFSesL4XhJbmT6oYrortamB9IK4nium84kZ0RsqXHx8dHoumkMoSpZceU5n5c6Fv/c8X0qhbXfvhfLVv/W7G2lM1RpWRDOmaQjAxLH2F45OnavukeCNOk7fyUkapYrQ1uhQajnDA7QXsjak1LZpiX+hP8fi9ZyVVA5xg9CZJrbnRKGlnyWh8NlIqJkWbJTQ4TrQaHNUBpF3PeSTDp6KjkFWm4UaLQcdEk50cEPkVDMjCfMs2UYg+wkJLSPHQ8g0nRmu/OEkuneMHq2tGW4laUHEjhF1vKltVxkoIkCocQz+Gn7A2yP3IldVzzFtXB9N2KK56NdORHcuqEdyDdbfbr8k+fKf80kF/N1wQvyInfz8mkO/VzL4M5y77xOag93hYSGKK46VVLSlGYmBVFr1jxkFW1ZAZ3C8dV8Dghr/xeUWzDCiXzaj6QVXPacNbi0pZtTnXmvqiRtdAGPaf7DsNj8nywVsi26tHKvGEQGvdI5FcTCJq/u5BtZb5UrIwTG0X8ltU9Mzy2oZiS2oYuBA6/FQUfIsuuUSSNk7VQc46znnOMdIhtWtRMa66TNIGDWewORHArug8s1iDu8sodMACEQOyAoK/ZNaQHgaxw0s9HP+lCKqRAEchsLYwdH2w6HsxPqJaVadg9esp9jSJQEStMz2onJnVCUduj6GKnO3CvVUWLro9DpztDNqaT87UDLZrLUNEVeZmR+RpF4AnKuo63ZTwEFrJlITdM8zILDM7s3yQZvafkXUpiBLuU5InfT6DCn/J4WJo1XQ2HNZkNMi4jFBFdgXe1UTGipRWaUMVrZsQtz8HRG5KUlCZZS5AAptJwnwOKac7L3BWB2H3lohwMwTXQ7sJWtMIIVuc4CYn1gixOTr5ZWDqIKgfz1C7in8GErdkBOgFL40FJBxp8aaFv1BKKT2s+4kgNwQ6IWpY5G5biaDbD3c5KoSLAZQzrj9bXiv/RC9hg9kn1fB8rIE7Xm/+R2cK/9oyQSZinFetrky1O9zFdM1OsZlo88p2M3yz2MUJJnLkj2aNyHyO6eifH2V4W0Sy3TSxWEpJYZ5eLxVlKFt9+dxUoX5zt9dIAaTPDtQGBzAZcFmkIJIhZ8C9MrnjdZdHvGjoN7E9YZaALsSjyCnFULAHQkQ2CsOZWABGaQJ1+5G3kNDOInI7awEH9eggXURFGvQ052jCllYOlMabdMI6KvmQRMrppHFKhc3bLBMBQjYUEsIiTCGAmmhCsuOnMKAsTMWbUxVcO8QW1J7oG7blD7w5yGUtQl9eyYM4jKA6qHRfLldE5OiIIvxH4PbxqSHzdcvPtaTywZB9ku0lNq2K0qW3pe2j6mIr9EqRxXnEGyGDxHSsubOIyGspFdJVsisN95dqgLy0kOnq7FkwD4+RkLAi+x6K2D/1Z1oBS4B6c9s4ZwmPssrL1nmvQBJGZ28i8Si7ngRa0F3s57UsdyTKysZkJTqGprSGiAVbRuPWiG9SrNGgf4xI8ZJPX6s8YnQbJWKBHFVtnb+zZO/Sl2BJHCb1TwsbkvYlxhpbQ9erYWyfaElII8jolvC1kKdplFvWmmn0ffU1PWMAO6wIPoHgNEZAzXQiR/cwgtg/q7BTgwWBmEjgVC4zPuVaa/ekTFDYmIJF+e9CGNxf3AgDjN5vgkM1vJVCTdx9/J6pvjYBLyDWvsCOwvTcYRqMwwy0Wk7+R+SHhA1XTawPSSCe1wMrpESRwJW1u4C9UWAVnoG0egkvuoZ/I5c3go62ixuhU7DaqYLjk/NKwtvcFdHvVei0gyaHx3EF2zYobaE800kN6X8OhrRqmbiCqrY37SUsOENuAI7URCBG4Q0dtrzamSgmkBXzD2kYmwnEOWZgEHM/Oc6ReS3Sv0UlDrc+VBbRrkpzFw2k6RPDGhbjgVIwzcEwW+zZJE6R9bYUtecuhAkE758H0nZ+Jk/1HaQW6ngwYAzDxqvZCCnSTq76qau5Cb9dtKfhAbzB0bRqkBKOU+C4O+rW8arOgwUunXWXjr+kMdpntHHfA6P2GHNzBfiOHC47tATAmsObgcub8DDlpUzrzCPD/so3BGWeRr7AQZD7ksyn6EaJtFywehwr+K9cfkD7/7Coy/eVfb95dfLj4NP9n/u9FtEvR2NBk0Qp4AL3R+8TfYX8gUK9rgdN2g9AfEQPDCDdas7YAT7n7gnKotEtHrbI5n5353iIv4R77MEzZi1WuG7gprBDtT+l8lwgb9BnePFz45/kteA5vtznUiuBVZaJx43XKHZKXXZ8FmIdY44pp3rKG21eJIR+zzQTd6U98uoIrdysqrIR6xaBTzYYnKeqGcQAj1EE8oolnooW+jfBeBa3O9QMUVLhC0hW/LwVcZ+3tHRVtVV4XpIfrraPZVXGfUe+/1v4N84eDMBD1V3rD1yEfdEiuSXaT+DjYQD5M4Ghn6JuSNZ8HeeBlOE0oORqv6fsD0knUxYqXfb0lsVb5uETfQvVu+ZsWrK7BV+9/jUdDUvIJQuB+RIUBRfBIlHsMG3f9VkmtL1qjZPfwHn7GOxLCsWP3DpyzOT0dL/cuJ+FODyVoyeN5OrUiL8k8aHRqEJ2OWY0GTC+I/rEnLBfpZG1Kgn2FfsfPwVY2DYB8F2shVelB1VqS2b+TJVpnznA3MT6sjft4Reanp/uwHU55tNwFQL5Usu/05enVZVSr6MoamG12knfCrLbTayXwxvhg86uGBJMdlOyIRTuyBR+Oq/VHn8ol5HqfjftP4KCi/7Rb7W11fGHP9ot1ydOrL/5gn86dN7Iv3gvndFE9vbBb+eL2chn5tegKVl88keMNH1XH4/PjyONnkOesevKvjuO6G0dXT8cpqeperzauh9AIb6snP9rAXffGEMrb1GtULkc0u+Xu+WvzlucAhwa3wb+I1dMHq7HnfJ0+s/qGIXtQOmbBGLu493ToKDL3laTkK1f1ZE3+Vmyigmfd7bZT7eRkCP9nXvj2JwgaEV78XBVzaXK3nSYzLZYw2/I7/E9QFu1MG6vNAhGECwikP4H0z3YihhtCJXhdYpnWGUYRPouCyGh8GMWBfQ+1s/b9Mtkj3zlsxW2feZgIsc7eXCeyqSrAZa/zL4UukaO3Evsjwyn5x9BenZMvNikwNSn5hLN4YfXoWdJoPcOOjiC3ctuQ5DneCqI8x1fLPI+GJ3n7hHn0J1BLAwQUAAAACABzZURd9VNgVSYQAABxNQAAGAAAAGNvbXBhcmVfY3JvcF9hYmxhdGlvbi5weZ0bZ3PcxvU7f8UGmgxx1B1YZCfWKfCMTFMZT9RGkv3ldAODwB4JEs0oIk8k/3vee1vRWMxMfMBi9/W6u3Ic57jIyrDirNhskigJ00VdpknDoqoo9yteJ985q9q8ZldJc160DUuLME7yM5YVMU9rVlSs4XXD4rAJvZ2d3+vwjC9ZuYXJOcvS/UiADxBeEJ6mYZMUuVdu2WJxGtY8TXK+iJOK/fL688nb396fsK87DP4WiyjM4wSgis/Hr9//+tuvr7+cwBegomwbGv508vHDpy87juPs7CRZWVQNC6szQFhz9X5RF7l6zsLmfGdTFRkr4SlNTpn88BE/7OzEfMMy3lRJVAc4LYiKfNPWQLEbZbMlUQa4PhVXNUOZhVHThilLw1MQxSsWFWmb5eJTWfE4iZDZ2mO/5wAZOI1ZBQyJCd95VXhIOAJNNszNi4YldZLXTZhHHBDOWZrUzQxFnHKigP3DZy9pgfqDb2G+7a+tiqveYhgRq9kGRuCNJTkDiGPAmm3J3W9h2vIZAGUEO2/wKw2y/7CDLhR6E9/gHVFJWeFfFSY1Z3/g15OqKirX0UINQB9Vcs2yFgzoFMQJ5F3D/+VwsQHcec7PwGa+IeiGn/GqdgTRdVuS6ny2qttMMNihak3TpB5A9GbiKln3xECvCVEf5mfcfTkTq5uiAf36DBdKhDOlMJQLfX8SrzDQhKAoWI4iQwuHr6wOszLlijcgOkpwFRIdZUAvkrxvmMFXIKHzDibI2YF3MMULwAzTtAdRMiXhWW8PQdscIqQjtscsaK4F4HmHulkP/PNx2i1znOSiaauc4VI3jKK2CqOtj+rRZAwWzoAyUtS8Y+34l4UQl4LNIUHYHOLMl3NFpy9/50Yfvn6aS3n64mcOAvEBgAwi0TmPLnkcyGDiViE4ZF20VcSla0gDsp0W5yBf9znPxrkRYO6WKlIZ98lZcXrBo0ZaEdAVtRBwyfYnwhog9c54MzRUZyaAoDQv+Rbl6TpK4M6cOUp0+CwFhY9CHI7FA3CKAEQYAY6ulh013M9iUteYbG4Awp3ToWiuoi96sObUSxqe1e696Bl7xt5CVIm2jF8j3ZDFMkh5xusg9bA3h16HTnTcJG+50Q2Piiom4QLUFSBZ20gtzQpCZUTuck8hACC4tiFIwCqEIzUUw+X4jPk+DQi4s4FV0x+uAk24UVrU3L2es62IcviEkvielBYmCWpmgKFLjtMqIA7W2syjtGn6E3RNOgYHgAqi4lwUHGxolnYUMHpXfkekhXN2KuUsJ4qcFs7IjCGRgLtCHSNli1WBl9SQoMF4YFKHZGuCAY7OnwZNkfqHfAEBIzyt1dvh0UwFAayUAqidXKhUwCuLaiuJ0u8gTKw8rAn0PayaZANSreH7zZ02+jzMuPBDlEpyhu4GQlZBBl/PwWAADD7y/FtSFXnG88b2Rix8AKwhYZ+BEhD0nYelkqNnNtW2qz1N1gqnrwEKLvCQzdpFuF7FgeOGXzcuz6MCy0TfaZvN4ifHNqzriJcNcz98JhuYW/YA+qgZx6cH7eY4zNHIECO7QeRoQbQU4gSjGo/eRCQkeQHBhgUlw/VENBafHxWQtSyBBIlpMiQrQih/iTf7i1fzBkwnbFMIyQ0gy4MzXqCCSaeQ0WIM31BLa0f4qwX06JY3TlhF56T6sojOyR5OwyY6D7CIx7ckO9PPNecx/qaVmJci13Eg6hAYGwQVWJFyKmhF9M+TDdT9QX0eHv34LxyK0rCuOaGts+KSByWvAhrEIcFMkkF7UI9BR0NWX5mTt1lwVVSXWO4JB1CZAGoxiNg8djXnCyU7iB8Q+XV9Jlc8XncKhVTSJuFpXINV7c7ZrndRJLkrZ8xUKpIsY0ykNSstBG1Wg7JeTOjX9mL00QU+OiAUFE1FEPAjBQc7SiiQfWiIDqxMoxQ4n2LikmmycVnOshy6G9bmyV/QAyAFNZJAfRHDLFE5s3v9bDUwqHWXvclpT1Bvdym2NlKfhjhKEwqXdKN1vwvqf4eG6PDxZIhVOkKURZ1gezNBQ9+TJokZTMQ27fFU9ZZr8qwGzFCoceqIsEYjguxzdo57AVZfo6rEV4yHVZrg1ybhzhMIUzg0SY/HMmVzVhaws+RaRntlc1PTnqBrs8xqckGmPCubbTc1iFKUInm/e7AIsfO9cBHXTuX2d5HOTTzEFkchWemSfU3+rzRqReEnsGn3sbIrHynjIKRa4AVZsl7pZmZVxEylZvndCqE4QYGSIU6+ztjPA399PGMKZifUHXqe9GHc20o56kwUw4qrU4wyNGcunmv4yuVzlAG378EE5mxx6B3M6VnXeHKVgIcxlEMq5LhrZNg+7HY3g80fWmvZMgUUMSp6PUIClmGFkkFCxu0RWiLlJlycHmcPF/VjQqS1DNg5hayu3QGkWvOopR0eqtoOHZOyVIc7dAmbHbAqZzZnY3EWSkL8uXM63YkEYjkB8Tb0jacxOuIG0TnuQdQsbCbpecbeYZHGmnPOqEji1S4UwdfgC2S7uH2a1LIpVeFtAeENUkeaRFvTpZKVYTcq9YZyWa9Mo95pT8Xkny377DL7sA3Lj3K8h3TQt2lnngpmY2tQLRLhk6KRgsriggsTz7SMdYZAwEorhtu7bodJdbqG7nfiLdicCC2+ahaseO9PJpmBq+kwIiiofTt+zS1N+JZSRqB0pec/RdBztren7F/vYcmde9z9d9VefQDcA9tqbx5fZThQM6yvYCS6B7YBAFN6vAtKhs8NmVS4aXgFMNRS07NZOIaNnFi+6rdPokSxOyiqrRHHxNxNm6aBOAK5r2DpLdaRzUKFgVxxQX5sgRb7V5IbaXxxstnwCnpo6i1kw4O7WNZunGBTNj3sVnAiX8d3hKR8cD2y1+OZ6HKtrTKBgEmMcpCwTIN/6A+ASfXgdhmSIRSAb6ZSsdi/T+7HpPi2opC7jwdQWNCTpy+Zw54z7CRF42ZBNGiMXXXygGVaI4lgSMYfY9UPoquB1+aK85wO0FSZZ2JB0FXzjcYB0liKyKNI9A2tnWBCCRCmzyyP8C0Gxicb+7DsSVrZFCJtZ5PQ7fZbSrhjTuOAe/Y1Bb3fxU5A02Y1BUdYGkET+wppkiUNaQ91sHK+QODFugiKsk5Cp0NRdBHcOHmF54ZV8Y1TZS+rKP4tiVGXeG51xiGNw+rvYnVTsJxfiTPSkc0P+HM+5OnWbCnbjYPYh6AzQxmSAT9ZOR3D8iYk09epjghjSEuTRGEKit00V6Hw5POwivHZkxKFkrFpkXER7Ey14fSj6aC/NC5hgNAks4NoydaD8pRXjQv1rvP53Yf/nbBPv79fsjIpRVCk+g6Enm5fASeqKuVVCXlYl4FhTYFzAe7EgBIgLMOK11OulaLJKYJ1dw7WcHTQtZl+xlWzLM1ga23nmPEFszEpJLma/qAofnt//OHdx7cnX06W7LSARomO2imD4HZ0fY5b+Eh+mYY5nh/LzkPGZqkvMIP75CI7qElNisjfTZuqCZzrZXZrSD2j++Lox8MXc/bDwb8P7j3vhQZnQUgt7lqYg9WYunXACJgogKGPRZi2+3WaRuBnIoouR+UdliXPY9f5NOZdMlQvgRrh+NT5Y2MN8RdcEpxBHKHp/G6cxBLwSC5ZvVjfS8/GOaatlhdjjQMQcqNh7srRXYR594piE0lE9xfybPd0y5z7kjNUyocHByDfcdBL72hzhxYUgWQQfAm5s6kVm2Z3CALlWresSR7z67nedzQNq7IcvSdqWQndFqCsR5uES3YjnHyJNrISz+sVgRYnumJIHH9Yx436zM46e4Qn6HZmdyOSQEiEcI54CJrrKGHAQvUI+dR1TFVmZdeZBVcwsXJinjahgwciN5S61bgBIBPTQn/SONfdg8MxpZkcPVxtiNHaUfZFBQRJ0LdU5Iut4709AUvma12d460TU5JTn9ctv/t9Q68QH+n5SDoC1Gplw8bz+gsUioVdjiHPF53D/8l7CEUEsYRud8g7AXgY6Sdzc8HAv1BnlEJA/sAwAetoVjYwJlderI3V+ANGxqGOlGpmhZCXr6QmRuUFlbkQiutC2D1CG4Xfn/D3aM5ezEZuSIik5Isf1a7KijmgGoLXgTiQ052A/6Vq+ZBwVZsEduyVIHyqf8Zj8hDSxER/CsBQvqPV7hCRkKTwSWOn0hO1WdP7w9ccZndDBNrnfP00VxZpLlnUvhx6RNM+XvV33bJjNX17mQ2RWLnHt55Vn5+F1WVcXOXQj2JUdcWPjNWIXRTGz9gxdrIjaRDF5OD+22cysyUElxsBBNILDe2u7/b2xLQudc6vSDZmO7NpkCU51FGKdY+9E5GfGJR36/oZirllOfNGEdyyTxDob9l/Fd237Fhv2+7366pb9ovZFbplr6UhwOM7tAP25pDdjqBYLBbsgf86JmGKYzGyNCv1WNnCrqbanHb0SJq0cG3VFKAbU03cylx6ByhvMIvuCneH1L7b7fVBHdacbmWL3/anqgioH3qAzSIF0OxR2aPKpXahyPjhn2ZcuZcel7WGsLvnYHjImLCSW/k/bVxkEkiFgc72GFQ4y+feD1jJlDC7zwry0F9vqBisl8btPHvGPvJqIU/HzI2LQTsH5Iq6TlneZ1nS3bKP+oIRfBMV2600qadbkby8KHlxRjszqsl8nLpy8Bm6dJifhhF3HUL69SsI3Ix9JV9mTneD4EF7hRdRAg0vN4kiDwnoGS8ZN12GJR3fyNKGgnFXBw7W14IG31eYxI3B7jqyoCnLteP7PUVjl8Chh9FFGdZxNUmBqaRxlLaf8FfsPxGnMzE2sHBlX29kKWMShzI+jER0123xs3WhFYKV2lMEm9PxU7vLmGk+2a6G2czptDV9AaGqd0W1hZIAgsWQJltFBRxU5mSPabOyB6WnLp/HJkT8bdPsSn6Doj9W3Km7x1K/SgF443upLhySm+nb3ktLI6L+Hg0LUv5Sj/sdNfaMBQ8ZkpErycp+JkALdfZgrRwYdtClXqrl645zy9ZAat00b0r/w6JeenLP14eWQJBHfAHZk7fLB/fGH/CQsQ5+VNob0+zL7TO5Laxj/33lLFieUvyff/5Jx+h9BHTzLW6zsnZVB9tJ0v1dzsfl/bs56SNv/KNBHYekWMnorSnkaHRtRObx60ZoYQEBq2j4nSNuBsGj7dpWLeisO6demAiEsgggqQWGdMEIH6T66Z9aVHR+L/7Zhfe6OmuR6Y/0xY15HVVJiVj8IIiLKAhm1kovjOMglEtcp/uPQoAxPLv28Y7kXF91o/7kXhidfzryd4GYf2WC8lXvDtbbdeM74jV4DAKAillO4qEfxKTOaTr3LIVu6EKZdfqGs73uERwNjRyeyduVZv9t5EqlpIRGyR3pSW4FqHYA74yPdwaaKc8Iwcsu4b8ukgz1uGghgZYEasHi0pKF21uHtaa1oSaurHhXVQI80UVSy9EEejxlrVuQTFhHSeK/CaEQsN2GDa6ePgZxFnfRKtanwJUV9B16FrZSUKQEAZYHQUB1ShCgnwSBPDoUTrPzf1BLAQIUABQAAAAIAHNlRF1moWWjFQsAAAcfAAAIAAAAAAAAAAAAAACAAQAAAAB0cmFpbi5weVBLAQIUABQAAAAIAHNlRF0AV1FtBAwAAIAiAAATAAAAAAAAAAAAAACAATsLAAB0cmFpbl9mdWxsX2NvbGFiLnB5UEsBAhQAFAAAAAgAc2VEXQGdElKhCQAAaRsAABcAAAAAAAAAAAAAAIABcBcAAHRyYWluX2dyb3VwX2Jhc2VsaW5lLnB5UEsBAhQAFAAAAAgAc2VEXfVTYFUmEAAAcTUAABgAAAAAAAAAAAAAAIABRiEAAGNvbXBhcmVfY3JvcF9hYmxhdGlvbi5weVBLBQYAAAAABAAEAAIBAACiMQAAAAA='

def digest(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def extract_safely(z, destination):
    destination.mkdir(parents=True, exist_ok=True)
    for member in z.infolist():
        assert (destination / member.filename).resolve().is_relative_to(destination.resolve()), '잘못된 ZIP 경로'
    z.extractall(destination)

source = UPLOAD / 'cropy_full_512.zip'
local = WORK / source.name
assert source.is_file(), f'기존 데이터 파일이 없습니다: {source}'
if not local.exists() or digest(local) != EXPECTED_DATA_SHA256:
    shutil.copyfile(source, local)
assert digest(local) == EXPECTED_DATA_SHA256, '데이터 ZIP이 기존 실험과 다릅니다.'
DATA = WORK / 'dataset'
if not (WORK / 'data_verified.json').exists():
    with zipfile.ZipFile(local) as z:
        assert shutil.disk_usage(WORK).free > sum(i.file_size for i in z.infolist()) + 2_000_000_000, '로컬 저장 공간 부족'
        extract_safely(z, WORK)
blob = base64.b64decode(CODE_B64)
assert hashlib.sha256(blob).hexdigest() == EXPECTED_CODE_SHA256
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    extract_safely(z, WORK / 'code')
assert digest(DATA / 'manifest.csv') == EXPECTED_MANIFEST_SHA256 == baseline_config['manifest_sha256']
with (DATA / 'manifest.csv').open(encoding='utf-8-sig') as f:
    rows = list(csv.DictReader(f))
assert Counter(r['split'] for r in rows) == {'1.Training': 32513, '2.Validation': 4070}
assert len({r['sha256'] for r in rows}) == len(rows) == 36583
assert len({r['packaged_sha256'] for r in rows}) == len(rows)
for i, row in enumerate(rows, 1):
    assert digest(DATA / row['image_path']) == row['packaged_sha256'], row['image_path']
    assert digest(DATA / row['label_path']) == row['label_sha256'], row['label_path']
    if i % 5000 == 0: print('검증:', i, '/', len(rows))
train_ds = torchvision.datasets.ImageFolder(DATA / 'images/train')
val_ds = torchvision.datasets.ImageFolder(DATA / 'images/val')
assert (len(train_ds), len(val_ds)) == (32513, 4070)
assert train_ds.classes == val_ds.classes == baseline_config['classes']
assert [int(c.split('_')[0]) for c in train_ds.classes] == list(range(9))
expected = dict(arch='resnet50', epochs=EPOCHS, batch_size=BATCH_SIZE, img_size=256,
    seed=42, lr=1e-4, balanced_sampler=True, selection='highest validation macro_f1; earliest tie',
    manifest_sha256=EXPECTED_MANIFEST_SHA256, classes=train_ds.classes, smoke_per_class=0,
    train_images=32513, val_images=4070, num_workers=2, train_geometry='random_crop')
assert dict(baseline_config, train_geometry=baseline_config.get('train_geometry', 'random_crop')) == expected, '기준 모델 설정이 예상과 다릅니다.'
(WORK / 'data_verified.json').write_text(json.dumps({'images': len(rows), 'manifest_sha256': EXPECTED_MANIFEST_SHA256}))
print('모든 데이터 검증 통과. 변경 변수: train_geometry random_crop -> full_resize')


## 20 epoch 학습 / 중단 후 재개
검증 Macro F1이 가장 높은 epoch를 선택하며 동률이면 앞선 epoch를 유지합니다. 각 epoch 끝에 Drive에 체크포인트를 저장합니다.
새 폴더에 다른 설정의 실행 결과가 있으면 자동으로 중단합니다. 기존 기준 모델을 덮어쓰지 않습니다. seed는 동일하게 고정하지만 전처리가 달라지므로 난수 소비 순서까지 동일하지는 않습니다.

In [ ]:
command = [sys.executable, '-u', str(WORK / 'code/train_full_colab.py'),
    '--data-dir', str(DATA / 'images'), '--output-dir', str(RUN),
    '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE), '--num-workers', '2',
    '--img-size', '256', '--arch', 'resnet50', '--train-geometry', 'full_resize']
if (RUN / 'last_checkpoint.pth').exists():
    command.append('--resume')
provenance = dict(torch=str(torch.__version__), torchvision=str(torchvision.__version__),
    python=platform.python_version(), gpu=torch.cuda.get_device_name(0),
    code_sha256=EXPECTED_CODE_SHA256, data_sha256=EXPECTED_DATA_SHA256,
    baseline=str(BASELINE), candidate=str(RUN), command=command)
(WORK / 'experiment_setup.json').write_text(json.dumps(provenance, ensure_ascii=False, indent=2))
print('이어서 학습:', '--resume' in command, '| 결과:', RUN)
with (WORK / 'training.log').open('a', encoding='utf-8') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        returncode = process.wait()
    except BaseException:
        process.terminate()
        process.wait()
        raise
if returncode:
    raise subprocess.CalledProcessError(returncode, command)
shutil.copyfile(WORK / 'experiment_setup.json', RUN / 'experiment_setup.json')
shutil.copyfile(WORK / 'training.log', RUN / 'training.log')


## 기존 최고 모델과 비교
전체 정확도·Macro F1, 9개 클래스의 정밀도·재현율·F1, 무노균병→무검은무늬병/배추정상과 역방향 오분류를 함께 비교합니다. 무노균병 재현율 하나만으로 모델을 교체하지 않습니다. 비교 파일은 새 결과 폴더에 저장됩니다.

In [ ]:
subprocess.run([sys.executable, str(WORK / 'code/compare_crop_ablation.py'),
    '--baseline-dir', str(BASELINE), '--candidate-dir', str(RUN),
    '--output-dir', str(RUN / 'comparison')], check=True)
from IPython.display import Markdown, display
display(Markdown((RUN / 'comparison/comparison.md').read_text(encoding='utf-8')))
print('새 최고 모델:', RUN / 'best_model.pth')
print('기존 모델:', BASELINE / 'best_model.pth')
